In [19]:
import pandas as pd
import numpy as np
import os

data_path = "~/LWCUP/results/merge_data/merge_data.parquet"

df = pd.read_parquet(os.path.expanduser(data_path))

df['date'].astype(int).unique().max()

np.int64(119)

In [14]:
# date列是纯数字，例如0，1，2，转化为日期，便于和time合并
df['Ndate'] = pd.to_datetime(df['date'].astype(int), unit='D', origin=pd.Timestamp('2020-01-01'))
df['timestamp'] = pd.to_datetime(df['Ndate'].dt.strftime('%Y-%m-%d') + ' ' + df['time'].astype(str), format='%Y-%m-%d %H:%M:%S')

0          09:40:00
1          09:40:03
2          09:40:06
3          09:40:09
4          09:40:12
             ...   
2401195    14:49:48
2401196    14:49:51
2401197    14:49:54
2401198    14:49:57
2401199    14:50:00
Name: time, Length: 2401200, dtype: object


In [15]:
df['timestamp']

0         2020-01-01 09:40:00
1         2020-01-01 09:40:03
2         2020-01-01 09:40:06
3         2020-01-01 09:40:09
4         2020-01-01 09:40:12
                  ...        
2401195   2020-01-10 14:49:48
2401196   2020-01-10 14:49:51
2401197   2020-01-10 14:49:54
2401198   2020-01-10 14:49:57
2401199   2020-01-10 14:50:00
Name: timestamp, Length: 2401200, dtype: datetime64[us]

In [2]:
df = df.sort_values(['sym', 'date']).set_index(['date', 'sym'])

In [3]:
null_counts = []
for sym in [f'{i}' for i in range(5)]:
    null_count = df.xs(sym, level='sym').isnull().sum()
    null_counts.append(null_count)

In [4]:
null_merge = pd.concat(null_counts, axis=1, keys=[f'sym_{i}' for i in range(5)])

null_merge['is_null'] = null_merge.sum(axis=1) > 0

null_merge[null_merge['is_null']]

,sym_0,sym_1,sym_2,sym_3,sym_4,is_null
ask1,0,2049,0,0,0,True
asize1,0,2049,0,0,0,True
ask2,0,2278,0,0,0,True
asize2,0,2278,0,0,0,True
ask3,0,2473,0,0,0,True
...,...,...,...,...,...,...
asize_rate6,0,2785,0,20,0,True
asize_rate7,0,2888,0,69,0,True
asize_rate8,0,2971,0,374,0,True
asize_rate9,0,3025,0,661,0,True


In [54]:
null_sym_dates = {}
for sym in [f'{i}' for i in range(5)]:
    sym_df = df.xs(f'{sym}', level='sym')
    dates = sym_df.groupby(by='date').apply(lambda x: x.isnull().sum()).sum(axis=1)[(sym_df.groupby(by='date').apply(lambda x: x.isnull().sum()).sum(axis=1) != 0) ].index.tolist()
    for date in dates:
        sym_date_df = sym_df[sym_df.index.get_level_values('date') == date]
        null_sym_dates.setdefault(sym, {})[date] = sym_date_df.loc[:, (sym_date_df.isnull().sum(axis=0) > 0)].columns.tolist()

In [55]:
null_sym_dates

{'1': {'42': ['ask1',
   'asize1',
   'ask2',
   'asize2',
   'ask3',
   'asize3',
   'ask4',
   'asize4',
   'ask5',
   'asize5',
   'ask6',
   'asize6',
   'ask7',
   'asize7',
   'ask8',
   'asize8',
   'ask9',
   'asize9',
   'ask10',
   'asize10',
   'spread1',
   'spread2',
   'spread3',
   'spread4',
   'spread5',
   'spread6',
   'spread7',
   'spread8',
   'spread9',
   'spread10',
   'ask_diff1',
   'ask_diff2',
   'ask_diff3',
   'ask_diff4',
   'ask_diff5',
   'ask_diff6',
   'ask_diff7',
   'ask_diff8',
   'ask_diff9',
   'ask_diff10',
   'ask_mean',
   'asize_mean',
   'ask_rate1',
   'ask_rate2',
   'ask_rate3',
   'ask_rate4',
   'ask_rate5',
   'ask_rate6',
   'ask_rate7',
   'ask_rate8',
   'ask_rate9',
   'ask_rate10',
   'asize_rate1',
   'asize_rate2',
   'asize_rate3',
   'asize_rate4',
   'asize_rate5',
   'asize_rate6',
   'asize_rate7',
   'asize_rate8',
   'asize_rate9',
   'asize_rate10'],
  '74': ['ask1',
   'asize1',
   'ask2',
   'asize2',
   'ask3',
   'a

In [56]:
import json
with open('/home1/zhuzhoufan/LWCUP/results/merge_data/null_sym_date_cols.json', 'w') as f:
    json.dump(null_sym_dates, f)

In [38]:
df.xs(('42', '1'))[df.xs(('42', '1'))['ask1'].isnull() == 0]

/tmp/ipykernel_476484/1358963701.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  df.xs(('42', '1'))[df.xs(('42', '1'))['ask1'].isnull() == 0]


time      open      high       low     close  volume_delta  \
date sym                                                                   
42   1    09:40:18  0.003343  0.100279  0.003343  0.095265      0.037541   
     1    09:40:21  0.003343  0.100279  0.003343  0.094708      0.010955   
     1    09:40:24  0.003343  0.100279  0.003343  0.089694      0.012514   
     1    09:40:27  0.003343  0.100279  0.003343  0.089694      0.000974   
     1    09:40:30  0.003343  0.100279  0.003343  0.085794      0.009057   
...            ...       ...       ...       ...       ...           ...   
     1    14:49:48  0.003343  0.100279  0.003343  0.045125      0.000073   
     1    14:49:51  0.003343  0.100279  0.003343  0.044568      0.000073   
     1    14:49:54  0.003343  0.100279  0.003343  0.044568      0.000049   
     1    14:49:57  0.003343  0.100279  0.003343  0.044568      0.001558   
     1    14:50:00  0.003343  0.100279  0.003343  0.044011      0.002410   

          amount_delta      bid1    bsize1      bid2  ...  asize_rate7  \
date sym                                              ...                
42   1       3043629.0  0.088022  0.000219  0.087465  ...          NaN   
     1        878792.0  0.089694  0.000170  0.088579  ...          NaN   
     1       1005233.0  0.089694  0.000073  0.088022  ...          NaN   
     1         78168.0  0.087465  0.000049  0.086908  ...          NaN   
     1        725343.0  0.084123  0.007620  0.082451  ...    -0.059135   
...                ...       ...       ...       ...  ...          ...   
     1          5627.0  0.044011  0.002459  0.043454  ...    -0.001193   
     1          5626.0  0.044011  0.002459  0.043454  ...     0.001193   
     1          3750.0  0.044011  0.002459  0.043454  ...     0.000000   
     1        120057.0  0.044011  0.002654  0.043454  ...     0.000000   
     1        185678.0  0.044011  0.002508  0.043454  ...    -0.001193   

          asize_rate8  asize_rate9  asize_rate10  midprice  label_5  label_10  \
date sym                                                                        
42   1            NaN          NaN           NaN  0.091643      0.0       0.0   
     1            NaN          NaN           NaN  0.091643      0.0       0.0   
     1            NaN          NaN           NaN  0.092201      0.0       0.0   
     1            NaN          NaN           NaN  0.088579      0.0       0.0   
     1            NaN          NaN           NaN  0.085237      0.0       0.0   
...               ...          ...           ...       ...      ...       ...   
     1            0.0    -0.000828      0.001315  0.044568      1.0       0.0   
     1            0.0     0.000828     -0.001315  0.044290      1.0       2.0   
     1            0.0     0.000000      0.000000  0.044290      1.0       2.0   
     1            0.0     0.000000      0.000000  0.044290      1.0       1.0   
     1            0.0    -0.000828      0.001315  0.044568      1.0       1.0   

          label_20  label_40  label_60  
date sym                                
42   1         0.0       0.0       0.0  
     1         0.0       0.0       0.0  
     1         0.0       0.0       0.0  
     1         0.0       0.0       0.0  
     1         0.0       0.0       0.0  
...            ...       ...       ...  
     1         0.0       0.0       0.0  
     1         1.0       0.0       0.0  
     1         1.0       0.0       0.0  
     1         0.0       0.0       0.0  
     1         0.0       0.0       0.0  

[3996 rows x 161 columns]